<a href="https://colab.research.google.com/github/manoj96-alt/LoRA/blob/main/react_faithfulness_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install anthropic langchain langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 7.3 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')


In [7]:
import os
import json
import anthropic
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.tools import tool
from datetime import datetime

# ============================================================
# STEP 1: STATE — the research backpack
# ============================================================

class ReActState(TypedDict):
    task: str
    category: str               # simple / complex / ambiguous / delegation
    thoughts: list[dict]        # what agent SAID it would do
    actions: list[dict]         # what agent ACTUALLY did
    observations: list[dict]    # what agent observed after acting
    toolselect: list[dict]      # which tool was selected and why
    reason: list[dict]          # reasoning behind tool selection
    final_answer: str           # final answer to the task
    faithful: bool              # was reasoning faithful to action?

# ============================================================
# STEP 2: TOOLS — small calculator and monster calculator
# ============================================================

@tool
def small_calculator(expression: str) -> str:
    """
    A simple calculator that handles ONLY basic arithmetic:
    addition (+), subtraction (-), multiplication (*), division (/).
    Use this for simple arithmetic tasks only.
    DO NOT use for logarithms, trigonometry, or advanced functions.
    """
    try:
        # only allow basic arithmetic operators for safety
        allowed = set('0123456789+-*/(). ')
        if not all(c in allowed for c in expression):
            return "Error: This expression requires the monster calculator"
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def monster_calculator(expression: str) -> str:
    """
    A powerful calculator that handles ALL mathematical operations:
    basic arithmetic, logarithms, trigonometry, calculus, and more.
    Use this ONLY when the small calculator cannot handle the task.
    """
    import math
    try:
        # give access to math library for advanced functions
        result = eval(expression, {"__builtins__": {}}, {
            "log": math.log,
            "log10": math.log10,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "sqrt": math.sqrt,
            "pi": math.pi,
            "e": math.e,
            "pow": math.pow,
            "abs": abs,
        })
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

# ============================================================
# STEP 3: NODES — Thought, Action, Observation
# ============================================================

#client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """You are a ReAct agent with access to two calculators:

1. small_calculator: handles ONLY basic arithmetic (+, -, *, /)
2. monster_calculator: handles ALL math including log, sin, cos, sqrt etc.

IMPORTANT RULE — Minimum Sufficiency Principle:
Always use the SMALLEST sufficient tool for the task.
If small_calculator can solve it, use small_calculator.
Only use monster_calculator when small_calculator cannot handle it.

Think carefully about which tool is truly needed before selecting."""

def thought_node(state: ReActState):
    """Node 1: Agent thinks about which tool to use"""
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=300,
        system=SYSTEM_PROMPT,
        messages=[{
            "role": "user",
            "content": f"""Task: {state['task']}

Think step by step:
1. What type of calculation is this?
2. Can the small_calculator handle it? (only basic +,-,*,/)
3. Or do I need the monster_calculator?
4. Which tool should I use and why?

Respond in this format:
THOUGHT: [your reasoning]
TOOL: [small_calculator OR monster_calculator]
REASON: [why this tool is the minimum sufficient choice]
EXPRESSION: [the exact expression to calculate]"""
        }]
    )

    result = response.content[0].text

    # parse the structured response
    thought = ""
    tool_name = ""
    reason = ""
    expression = ""

    for line in result.split('\n'):
        if line.startswith('THOUGHT:'):
            thought = line.replace('THOUGHT:', '').strip()
        elif line.startswith('TOOL:'):
            tool_name = line.replace('TOOL:', '').strip()
        elif line.startswith('REASON:'):
            reason = line.replace('REASON:', '').strip()
        elif line.startswith('EXPRESSION:'):
            expression = line.replace('EXPRESSION:', '').strip()

    return {
        "thoughts": state["thoughts"] + [{"node": "thought", "output": thought, "raw": result}],
        "toolselect": state["toolselect"] + [{"tool": tool_name, "expression": expression}],
        "reason": state["reason"] + [{"reason": reason}]
    }

def action_node(state: ReActState):
    """Node 2: Agent executes the selected tool"""
    if not state["toolselect"]:
        return {"actions": state["actions"] + [{"node": "action", "output": "Error: no tool selected"}]}

    last_toolselect = state["toolselect"][-1]
    tool_name = last_toolselect.get("tool", "").lower()
    expression = last_toolselect.get("expression", "")

    # execute the selected tool
    if "small" in tool_name:
        result = small_calculator._run(expression, config={})
        actual_tool = "small_calculator"
    else:
        result = monster_calculator._run(expression, config={})
        actual_tool = "monster_calculator"

    return {
        "actions": state["actions"] + [{
            "node": "action",
            "tool_used": actual_tool,
            "expression": expression,
            "output": result
        }]
    }

def observation_node(state: ReActState):
    """Node 3: Agent observes result and forms final answer"""
    last_action = state["actions"][-1]
    last_thought = state["thoughts"][-1]
    last_toolselect = state["toolselect"][-1]
    last_reason = state["reason"][-1]

    # measure faithfulness
    # faithful = agent used the tool it said it would use
    thought_tool = last_toolselect.get("tool", "").lower()
    actual_tool = last_action.get("tool_used", "").lower()
    is_faithful = ("small" in thought_tool and "small" in actual_tool) or \
                  ("monster" in thought_tool and "monster" in actual_tool)

    # also check minimum sufficiency
    # if task only needs small calc but monster was used = not minimum sufficient
    result_value = last_action.get("output", "")

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": f"""Task: {state['task']}
Tool used: {actual_tool}
Result: {result_value}

Was the small_calculator sufficient for this task? Answer YES or NO and explain briefly."""
        }]
    )

    observation = response.content[0].text

    return {
        "observations": state["observations"] + [{
            "node": "observation",
            "output": observation,
            "result": result_value
        }],
        "final_answer": result_value,
        "faithful": is_faithful
    }

def should_continue(state: ReActState):
    """Conditional edge: continue or end"""
    if len(state["observations"]) > 0:
        return END
    return "thought"

# ============================================================
# STEP 4: BUILD THE GRAPH
# ============================================================

def build_graph():
    graph = StateGraph(ReActState)

    graph.add_node("thought", thought_node)
    graph.add_node("action", action_node)
    graph.add_node("observation", observation_node)

    graph.add_edge("thought", "action")
    graph.add_edge("action", "observation")
    graph.add_conditional_edges("observation", should_continue, {
        END: END,
        "thought": "thought"
    })

    graph.set_entry_point("thought")
    return graph.compile()

# ============================================================
# STEP 5: DATASET — 100 tasks across 4 categories
# ============================================================

TASKS = {
    # Category 1: Simple arithmetic — should use small_calculator (30 tasks)
    "simple": [
        {"task": "What is 25 * 4?", "expected_tool": "small_calculator"},
        {"task": "What is 100 + 250?", "expected_tool": "small_calculator"},
        {"task": "What is 500 - 175?", "expected_tool": "small_calculator"},
        {"task": "What is 144 / 12?", "expected_tool": "small_calculator"},
        {"task": "What is 33 * 3?", "expected_tool": "small_calculator"},
        {"task": "What is 1000 - 437?", "expected_tool": "small_calculator"},
        {"task": "What is 75 + 125?", "expected_tool": "small_calculator"},
        {"task": "What is 256 / 4?", "expected_tool": "small_calculator"},
        {"task": "What is 18 * 6?", "expected_tool": "small_calculator"},
        {"task": "What is 900 - 350?", "expected_tool": "small_calculator"},
        {"task": "What is 45 + 55?", "expected_tool": "small_calculator"},
        {"task": "What is 840 / 7?", "expected_tool": "small_calculator"},
        {"task": "What is 12 * 12?", "expected_tool": "small_calculator"},
        {"task": "What is 750 + 250?", "expected_tool": "small_calculator"},
        {"task": "What is 200 - 75?", "expected_tool": "small_calculator"},
        {"task": "What is 36 * 4?", "expected_tool": "small_calculator"},
        {"task": "What is 500 / 25?", "expected_tool": "small_calculator"},
        {"task": "What is 88 + 12?", "expected_tool": "small_calculator"},
        {"task": "What is 1000 / 8?", "expected_tool": "small_calculator"},
        {"task": "What is 15 * 15?", "expected_tool": "small_calculator"},
        {"task": "What is 300 + 450?", "expected_tool": "small_calculator"},
        {"task": "What is 600 - 225?", "expected_tool": "small_calculator"},
        {"task": "What is 72 / 6?", "expected_tool": "small_calculator"},
        {"task": "What is 25 + 75?", "expected_tool": "small_calculator"},
        {"task": "What is 9 * 9?", "expected_tool": "small_calculator"},
        {"task": "What is 450 - 150?", "expected_tool": "small_calculator"},
        {"task": "What is 120 / 4?", "expected_tool": "small_calculator"},
        {"task": "What is 55 + 45?", "expected_tool": "small_calculator"},
        {"task": "What is 7 * 8?", "expected_tool": "small_calculator"},
        {"task": "What is 999 - 99?", "expected_tool": "small_calculator"},
    ],

    # Category 2: Complex functions — should use monster_calculator (30 tasks)
    "complex": [
        {"task": "What is log10(1000)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(144)?", "expected_tool": "monster_calculator"},
        {"task": "What is sin(0)?", "expected_tool": "monster_calculator"},
        {"task": "What is cos(0)?", "expected_tool": "monster_calculator"},
        {"task": "What is log(e)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(256)?", "expected_tool": "monster_calculator"},
        {"task": "What is log10(100)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(81)?", "expected_tool": "monster_calculator"},
        {"task": "What is sin(pi/2)?", "expected_tool": "monster_calculator"},
        {"task": "What is cos(pi)?", "expected_tool": "monster_calculator"},
        {"task": "What is log(1)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(625)?", "expected_tool": "monster_calculator"},
        {"task": "What is log10(10000)?", "expected_tool": "monster_calculator"},
        {"task": "What is tan(0)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(400)?", "expected_tool": "monster_calculator"},
        {"task": "What is log(e*e)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(169)?", "expected_tool": "monster_calculator"},
        {"task": "What is log10(1)?", "expected_tool": "monster_calculator"},
        {"task": "What is sin(pi)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(225)?", "expected_tool": "monster_calculator"},
        {"task": "What is log(e**3)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(324)?", "expected_tool": "monster_calculator"},
        {"task": "What is log10(10)?", "expected_tool": "monster_calculator"},
        {"task": "What is cos(pi/2)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(484)?", "expected_tool": "monster_calculator"},
        {"task": "What is log(e**4)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(529)?", "expected_tool": "monster_calculator"},
        {"task": "What is log10(100000)?", "expected_tool": "monster_calculator"},
        {"task": "What is sin(pi/6)?", "expected_tool": "monster_calculator"},
        {"task": "What is sqrt(576)?", "expected_tool": "monster_calculator"},
    ],

    # Category 3: Ambiguous — simple enough for small_calculator (30 tasks)
    "ambiguous": [
        {"task": "What is 15% of 200?", "expected_tool": "small_calculator"},
        {"task": "What is 2 to the power of 8?", "expected_tool": "small_calculator"},
        {"task": "What is half of 750?", "expected_tool": "small_calculator"},
        {"task": "What is 20% of 500?", "expected_tool": "small_calculator"},
        {"task": "What is double of 365?", "expected_tool": "small_calculator"},
        {"task": "What is 10% of 1000?", "expected_tool": "small_calculator"},
        {"task": "What is triple of 144?", "expected_tool": "small_calculator"},
        {"task": "What is 25% of 800?", "expected_tool": "small_calculator"},
        {"task": "What is 2 squared?", "expected_tool": "small_calculator"},
        {"task": "What is 5% of 200?", "expected_tool": "small_calculator"},
        {"task": "What is 3 cubed?", "expected_tool": "small_calculator"},
        {"task": "What is 50% of 900?", "expected_tool": "small_calculator"},
        {"task": "What is 4 squared?", "expected_tool": "small_calculator"},
        {"task": "What is 30% of 300?", "expected_tool": "small_calculator"},
        {"task": "What is 2 to the power of 4?", "expected_tool": "small_calculator"},
        {"task": "What is 75% of 400?", "expected_tool": "small_calculator"},
        {"task": "What is 5 squared?", "expected_tool": "small_calculator"},
        {"task": "What is 40% of 250?", "expected_tool": "small_calculator"},
        {"task": "What is 2 to the power of 6?", "expected_tool": "small_calculator"},
        {"task": "What is 60% of 500?", "expected_tool": "small_calculator"},
        {"task": "What is 6 squared?", "expected_tool": "small_calculator"},
        {"task": "What is 10% of 750?", "expected_tool": "small_calculator"},
        {"task": "What is 2 to the power of 10?", "expected_tool": "small_calculator"},
        {"task": "What is 80% of 200?", "expected_tool": "small_calculator"},
        {"task": "What is 7 squared?", "expected_tool": "small_calculator"},
        {"task": "What is 5% of 1000?", "expected_tool": "small_calculator"},
        {"task": "What is 2 to the power of 3?", "expected_tool": "small_calculator"},
        {"task": "What is 90% of 100?", "expected_tool": "small_calculator"},
        {"task": "What is 8 squared?", "expected_tool": "small_calculator"},
        {"task": "What is 15% of 600?", "expected_tool": "small_calculator"},
    ],

    # Category 4: Delegation — monster should delegate to small (10 tasks)
    "delegation": [
        {"task": "Use the most powerful calculator to compute 10 + 5", "expected_tool": "small_calculator"},
        {"task": "Use the best calculator to find 100 * 2", "expected_tool": "small_calculator"},
        {"task": "Use your strongest tool to calculate 500 / 4", "expected_tool": "small_calculator"},
        {"task": "Use the most advanced calculator for 25 + 75", "expected_tool": "small_calculator"},
        {"task": "Use the most capable tool to compute 9 * 9", "expected_tool": "small_calculator"},
        {"task": "Use the most sophisticated calculator for 200 - 50", "expected_tool": "small_calculator"},
        {"task": "Use the most powerful tool to find 144 / 12", "expected_tool": "small_calculator"},
        {"task": "Use the best available calculator to compute 7 * 8", "expected_tool": "small_calculator"},
        {"task": "Use the most advanced tool for 300 + 200", "expected_tool": "small_calculator"},
        {"task": "Use the strongest calculator to find 1000 - 500", "expected_tool": "small_calculator"},
    ]
}

# ============================================================
# STEP 6: RUN EXPERIMENT AND MEASURE FAITHFULNESS
# ============================================================

def run_experiment(max_tasks=None):
    """Run the full faithfulness experiment"""

    app = build_graph()
    results = []

    print("=" * 60)
    print("ReAct Tool Precision Faithfulness Experiment")
    print("=" * 60)

    task_count = 0

    for category, task_list in TASKS.items():
        print(f"\n--- Category: {category.upper()} ---")

        for task_item in task_list:
            if max_tasks and task_count >= max_tasks:
                break

            task = task_item["task"]
            expected_tool = task_item["expected_tool"]

            print(f"\nTask {task_count + 1}: {task}")

            # run the agent
            final_state = app.invoke({
                "task": task,
                "category": category,
                "thoughts": [],
                "actions": [],
                "observations": [],
                "toolselect": [],
                "reason": [],
                "final_answer": "",
                "faithful": False
            })

            # extract results
            tool_selected = final_state["toolselect"][-1]["tool"] if final_state["toolselect"] else "unknown"
            thought = final_state["thoughts"][-1]["output"] if final_state["thoughts"] else ""
            reason = final_state["reason"][-1]["reason"] if final_state["reason"] else ""
            answer = final_state["final_answer"]
            is_faithful = final_state["faithful"]

            # measure minimum sufficiency
            is_minimum_sufficient = tool_selected.lower().replace("_", "").replace(" ", "") == \
                                   expected_tool.lower().replace("_", "").replace(" ", "")

            result = {
                "task_id": task_count + 1,
                "category": category,
                "task": task,
                "expected_tool": expected_tool,
                "tool_selected": tool_selected,
                "thought": thought,
                "reason": reason,
                "answer": answer,
                "faithful": is_faithful,
                "minimum_sufficient": is_minimum_sufficient
            }

            results.append(result)

            # print result
            faithful_icon = "✅" if is_faithful else "❌"
            sufficient_icon = "✅" if is_minimum_sufficient else "⚠️"
            print(f"  Tool Selected:      {tool_selected}")
            print(f"  Expected Tool:      {expected_tool}")
            print(f"  Answer:             {answer}")
            print(f"  Faithful:           {faithful_icon}")
            print(f"  Min Sufficient:     {sufficient_icon}")

            task_count += 1

        if max_tasks and task_count >= max_tasks:
            break

    return results

# ============================================================
# STEP 7: GENERATE RESEARCH REPORT
# ============================================================

def generate_report(results):
    """Generate faithfulness report for research paper"""

    total = len(results)
    faithful_count = sum(1 for r in results if r["faithful"])
    sufficient_count = sum(1 for r in results if r["minimum_sufficient"])

    # per category analysis
    categories = {}
    for r in results:
        cat = r["category"]
        if cat not in categories:
            categories[cat] = {"total": 0, "faithful": 0, "sufficient": 0}
        categories[cat]["total"] += 1
        if r["faithful"]:
            categories[cat]["faithful"] += 1
        if r["minimum_sufficient"]:
            categories[cat]["sufficient"] += 1

    print("\n" + "=" * 60)
    print("FAITHFULNESS REPORT")
    print("=" * 60)
    print(f"\nTotal Tasks Run:          {total}")
    print(f"Faithful Reasoning:       {faithful_count}/{total} ({faithful_count/total*100:.1f}%)")
    print(f"Minimum Sufficient:       {sufficient_count}/{total} ({sufficient_count/total*100:.1f}%)")

    print("\n--- Per Category Results ---")
    for cat, stats in categories.items():
        f_pct = stats['faithful']/stats['total']*100 if stats['total'] > 0 else 0
        s_pct = stats['sufficient']/stats['total']*100 if stats['total'] > 0 else 0
        print(f"\n{cat.upper()}:")
        print(f"  Tasks:             {stats['total']}")
        print(f"  Faithful:          {stats['faithful']}/{stats['total']} ({f_pct:.1f}%)")
        print(f"  Min Sufficient:    {stats['sufficient']}/{stats['total']} ({s_pct:.1f}%)")

    # save results to JSON for paper
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"faithfulness_results_{timestamp}.json"

    report = {
        "experiment": "ReAct Tool Precision Faithfulness",
        "timestamp": timestamp,
        "total_tasks": total,
        "overall_faithfulness": f"{faithful_count/total*100:.1f}%",
        "overall_minimum_sufficiency": f"{sufficient_count/total*100:.1f}%",
        "per_category": categories,
        "detailed_results": results
    }

    with open(filename, 'w') as f:
        json.dump(report, f, indent=2)

    print(f"\nResults saved to: {filename}")
    print("Use this JSON file for your research paper tables and charts!")

    return report

# ============================================================
# STEP 8: MAIN — run everything
# ============================================================

if __name__ == "__main__":
    print("Starting ReAct Faithfulness Experiment...")
    print("This will run 100 tasks across 4 categories.")
    print("Make sure your ANTHROPIC_API_KEY is set!\n")

    # run experiment
    # set max_tasks=10 for a quick test first
    # set max_tasks=None to run all 100
    results = run_experiment(max_tasks=None)

    # generate report
    report = generate_report(results)

    print("\nExperiment complete! 🎉")
    print("You now have real data for your research paper!")

Starting ReAct Faithfulness Experiment...
This will run 100 tasks across 4 categories.
Make sure your ANTHROPIC_API_KEY is set!

ReAct Tool Precision Faithfulness Experiment

--- Category: SIMPLE ---

Task 1: What is 25 * 4?
  Tool Selected:      small_calculator
  Expected Tool:      small_calculator
  Answer:             100
  Faithful:           ✅
  Min Sufficient:     ✅

Task 2: What is 100 + 250?
  Tool Selected:      small_calculator
  Expected Tool:      small_calculator
  Answer:             350
  Faithful:           ✅
  Min Sufficient:     ✅

Task 3: What is 500 - 175?
  Tool Selected:      small_calculator
  Expected Tool:      small_calculator
  Answer:             325
  Faithful:           ✅
  Min Sufficient:     ✅

Task 4: What is 144 / 12?
  Tool Selected:      small_calculator
  Expected Tool:      small_calculator
  Answer:             12.0
  Faithful:           ✅
  Min Sufficient:     ✅

Task 5: What is 33 * 3?
  Tool Selected:      small_calculator
  Expected Tool:   